# Persisting memory across Strands Agents sessions

In this example you will learn how to persist memory across different sessions in your Strands Agents. 

We will use the use case of an agent that does web search using a `duckduckgo` search API.

In this notebook, we will:
- Explore the capabilities of a memory-powered Strands agent.
- Learn how to store, retrieve, and list memories.
- Understand how to perform web searches via the agent.
- Interact with the agent in an interactive loop.


### Usage Examples

Storing memories:
```
Remember that I prefer tea over coffee
```

Retrieving memories:
```
What do I prefer to drink?
```

Listing all memories:
```
Show me everything you remember about me
```

### Tips for Memory Usage

- Be explicit when asking the agent to remember information
- Use specific queries to retrieve relevant memories
- Memory persistence enables more natural and contextual conversations

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account and AWS credentials configured in the environment
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock
* IAM role with permissions to create Amazon Bedrock Knowledge Base, Amazon S3 bucket and Amazon DynamoDB

Let's now install the requirement packages for our Strands Agent

In [ ]:
# Install the required packages
!pip install -r requirements.txt

In [ ]:

# Import Required Libraries
import os
from strands import Agent, tool
from strands.models import bedrock
import time

from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException


os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
bedrock.DEFAULT_BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0" #Optional: Set a default model for Bedrock
os.environ["MEM0_USER_ID"] = "name_of_user" #to change with username
os.environ["MEM0_LLM_PROVIDER"] = "aws_bedrock"
os.environ["MEM0_LLM_MODEL"] = (
    "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
)

## Mem0 Configuration

### Memory Backend Options

The Mem0 Memory Tool supports three different backend configurations:

1. **[OpenSearch](https://aws.amazon.com/opensearch-service/features/serverless/)** (Recommended for AWS environments):
   - Requires AWS credentials and OpenSearch configuration
   - Set `OPENSEARCH_HOST` and optionally `AWS_REGION` (defaults to us-west-2)

2. **[FAISS]((https://faiss.ai/index.html))** (Default for local development):
   - Uses FAISS as the local vector store backend
   - Requires faiss-cpu package for local vector storage
   - No additional configuration needed

3. **Mem0 Platform**:
   - Uses the [Mem0 Platform API](https://docs.mem0.ai/platform/quickstart) for memory management
   - Requires a Mem0 API key : `MEM0_API_KEY` in the environment variables


### Environment Configuration

| Environment Variable | Description | Default | Required For |
|---------------------|-------------|----------|--------------|
| OPENSEARCH_HOST | OpenSearch Serverless Host URL | None | OpenSearch |
| AWS_REGION | AWS Region for OpenSearch | us-west-2 | OpenSearch |
| MEM0_API_KEY | Mem0 Platform API key | None | Mem0 Platform |
| DEV | Enable development mode | false | All modes |


For the scope of this lab, we can use 2 options as a backend for memory management:
### Option 1. [Opensearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/) 

This will be our setup architecture for AOSS: 

<div style="text-align:left">
    <img src="images/arch_AOSS.png" width="65%" />
</div>



In [ ]:
# You can manually define your Opensearch Host 
#os.environ["OPENSEARCH_HOST"] = "<your-opensearch-host>.<region>.aoss.amazonaws.com"

In [ ]:
# OR - Run the script to Create Opensearch Serverless resource in your AWS Account
!bash prereqs/deploy_OSS.sh

In [ ]:
wait_time = 120
print(f"Waiting {wait_time} seconds for OpenSearch Serverless collection to be fully operational...")
time.sleep(wait_time)


In [ ]:
# Option 1: Opensearch Serverless
from dotenv import load_dotenv
load_dotenv() # take Opensearch environment variables


### Option 2 [Mem0 Platform](https://docs.mem0.ai/platform):

#### [NOTE]: This is not Needed if you have already deployed the Opensearch Serverless option.

As an alternative, you can create Mem0 API keys by following the steps [here](https://docs.mem0.ai/platform/quickstart#2-api-key-setup) and add it as an environment variable **MEM0_API_KEY**.
This would be the architecture of the setup for Mem0 Platform:

<div style="text-align:left">
    <img src="images/arch_mem0.png" width="65%" />
</div>


To enable the agent's functionality, we need to configure environment variables for AWS credentials and OpenSearch / Mem0 Platform. These variables are used for memory storage and retrieval.

In [ ]:
# Option 2: Mem0 API key

#os.environ["MEM0_API_KEY"] = "<your-mem0-api-key>"

## Define System Prompt

The `SYSTEM_PROMPT` variable defines the behavior and capabilities of the memory agent. This prompt guides the agent to provide personalized responses based on stored memories and perform web searches when necessary.

In [ ]:
# Define a focused system prompt for memory operations
SYSTEM_PROMPT = """You are a helpful personal assistant for a user. Your task is to assist the user by providing personalized responses based on their history. 

Capabilities:
- You can store information using the mem0_memory tool (action="store").
- You can retrieve relevant memories using the mem0_memory tool (action="retrieve").
- You can use the websearch tool to find information on the web.

Key Rules:
- Be conversational and natural in your responses.
- Always retrieve memories before responding to the user and use them to inform your response.
- Store any new user information and user preferences in mem0_memory.
- Only share relevant information.
- Politely indicate when you don't have the information.
"""

## Define Web Search Tool

The `websearch` tool using [Duckduckgo Search API](https://github.com/deedy5/duckduckgo_search) function allows the agent to perform web searches. This function handles exceptions and returns search results or appropriate error messages.

In [ ]:
@tool
def websearch(
    keywords: str,
    region: str = "us-en",
    max_results: int | None = None,
) -> str:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"

## Create Memory Agent

We will now initialize the memory-focused agent using the defined tools and system prompt. The Strands agent is capable of:
1. Storing and retrieving memories based on context. It uses memory to create more personalized and contextual AI interactions.
2. Performing web searches using DuckDuckGo to give updated information.

In [ ]:
from strands_tools import mem0_memory
from strands_tools.mem0_memory import Mem0ServiceClient

print(
    Mem0ServiceClient.DEFAULT_CONFIG["llm"]
)


In [ ]:
# Create an agent with memory, websearch tool
USER_ID = "new_user" # Replace with actual user ID

memory_agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
    tools=[mem0_memory, websearch],
)

## Demonstrate Memory Operations

The following examples demonstrate how to store, retrieve, and list memories using the memory agent.

- **store**: Save important information for later retrieval
  - Store user preferences
  - Remember important facts
  - Maintain conversation context

- **retrieve**: Access relevant memories based on queries
  - Find previously stored information
  - Provide personalized responses based on user history

- **list**: View all stored memories
  - See what information has been retained
  - Audit stored memories

In [ ]:
# Store initial memories to demonstrate retrieval
memory_agent.tool.mem0_memory(
    action="store", content=f"The user's name is {USER_ID}.", user_id=USER_ID
)
memory_agent.tool.mem0_memory(
    action="store", 
    content="I like to drink tea more than coffee.", 
    user_id=USER_ID
)

In [ ]:
# Retrieve memories
retrieved_memories = memory_agent.tool.mem0_memory(
    action="retrieve", query="What is the user's name?", user_id=USER_ID
)
print("Retrieved Memories:", retrieved_memories)

In [ ]:
# Retrieve memories about preferences
memory_agent.tool.mem0_memory(
    action="retrieve",
    query="What are the my drink preferences?",
    user_id=USER_ID
)

In [ ]:
# Ask the agent a question
response = memory_agent("What are the events happening in the New York today?")
print(response)

In [ ]:

# List all stored memories
print("All Stored Memories:")
all_memories = memory_agent.tool.mem0_memory(
    action="list", user_id=USER_ID
)

## Interactive Agent Usage

Finally, we provide an interactive loop for users to interact with the memory agent. Users can store new memories, retrieve existing ones, or list all stored memories.

To test interactive usage: 
1. Install the requirements: `pip install -r requirements.txt`
1. Run the python file `personal_agent_with_memory.py`. 

## Conclusion

This notebook demonstrates how to create a personal agent with memory capabilities using the Strands framework. The agent can:

1. Store information about the user
2. Retrieve relevant memories based on context
3. Search the web for additional information
4. Provide personalized responses

By combining these capabilities, the agent can maintain context across conversations and provide more personalized assistance over time.

### Cleanup

Run this bash script to clean up the Opensearch Serverless resources. You don't need to run this if you used the "MEM0_PLATFORM_API".

In [ ]:
!sh prereqs/cleanup_OSS.sh

---

# Exercise — Prove the memory is real

**Time: 10-15 minutes.**

Here is the trap this lab sets. Tell an agent *"I am allergic to peanuts"*, then ask it what to avoid.
It answers correctly. You conclude the memory works.

It probably did not use memory at all. The allergy was still in the conversation, three messages up,
so the model just read it. Nothing was written to the store. Restart the kernel and the agent has
forgotten you, which is the one thing memory was supposed to prevent.

There is only one honest test: **a brand new agent, with an empty conversation, must still know.**

**Write two things:**

1. The `SYSTEM_PROMPT` for a memory-using agent. It has the `mem0_memory` tool; your prompt decides
   whether it bothers to `store` new facts and `retrieve` before answering.
2. The agent itself.

The self-check tells your agent the fact, then throws that agent away, builds a fresh one, and asks
the question. It also reads the memory store directly. Both have to hold.

In [ ]:
# ---- provided scaffold ----
import os
import uuid

os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
os.environ["MEM0_LLM_PROVIDER"] = "aws_bedrock"
os.environ["MEM0_LLM_MODEL"] = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# A user id of your own, so your memories do not mix with anyone else's.
EX_USER_ID = f"student_{uuid.uuid4().hex[:8]}"
os.environ["MEM0_USER_ID"] = EX_USER_ID

from strands import Agent
from strands_tools import mem0_memory

MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

FACT = "Please remember this about me: I am allergic to peanuts."
QUESTION = "Suggest a snack for me. Is there anything you should avoid?"


def stored_memories():
    """Read the memory store directly, without going through your agent."""
    reader = Agent(model=MODEL, system_prompt="You manage memory.", tools=[mem0_memory])
    result = reader.tool.mem0_memory(action="list", user_id=EX_USER_ID)
    return str(result)


print("your user id:", EX_USER_ID)

In [ ]:
# ---------------------------------------------------------------------------
# YOUR WORK GOES HERE
#
# Write the system prompt so the agent stores what it is told and retrieves
# before it answers. Then build the agent.
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """TODO: write the system prompt."""

if "TODO" in SYSTEM_PROMPT:
    raise NotImplementedError("Write the system prompt, then re-run this cell.")


def build_agent():
    """Return a NEW agent with no conversation history. Used twice below."""
    return Agent(model=MODEL, system_prompt=SYSTEM_PROMPT, tools=[mem0_memory])


# Session 1: tell the agent the fact, then discard it entirely.
session_one = build_agent()
session_one(FACT)
del session_one

# Session 2: a brand new agent, empty history. It can only know via the store.
session_two = build_agent()
fresh_answer = str(session_two(QUESTION))
print("\n--- what the fresh agent replied ---")
print(fresh_answer[:400])

In [ ]:
memories = stored_memories()

checks = {
    "the allergy was actually written to the memory store":
        "peanut" in memories.lower(),
    "a brand new agent, with no conversation history, knew about it":
        "peanut" in fresh_answer.lower(),
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

print("\n--- what is in the store ---")
print(memories[:400])

print()
if all(checks.values()):
    print("Exercise complete.")
elif "peanut" not in memories.lower():
    print("Nothing was stored. Your prompt is not telling the agent to write facts down.")
else:
    print("It is stored but the new agent did not look. Tell it to retrieve before answering.")

## Stuck?

One possible solution is stored below, encoded so you do not read it by accident. Try the exercise
first: the point is the attempt, not the answer.

Run `show_solution()` to print it, or `load_solution()` to run it, then re-run the self-check.

In [ ]:
import base64

_SOLUTION = (
    "U1lTVEVNX1BST01QVCA9ICIiIllvdSBhcmUgYSBwZXJzb25hbCBhc3Npc3RhbnQgd2l0aCBhIHBlcnNpc3RlbnQgbWVt"
    "b3J5LgoKWW91IGhhdmUgdGhlIG1lbTBfbWVtb3J5IHRvb2wuIFVzZSBpdCBhcyBmb2xsb3dzLCBldmVyeSBzaW5nbGUg"
    "dGltZToKCjEuIEJlZm9yZSB5b3UgYW5zd2VyIGFueXRoaW5nLCBjYWxsIG1lbTBfbWVtb3J5IHdpdGggYWN0aW9uPSJy"
    "ZXRyaWV2ZSIgYW5kIGEgcXVlcnkKICAgZGVzY3JpYmluZyB3aGF0IHlvdSBuZWVkIHRvIGtub3cgYWJvdXQgdGhlIHVz"
    "ZXIuIE5ldmVyIGFuc3dlciBmcm9tIHRoZSBjb252ZXJzYXRpb24KICAgYWxvbmU6IHRoZSBjb252ZXJzYXRpb24gaXMg"
    "bG9zdCBiZXR3ZWVuIHNlc3Npb25zLCB0aGUgc3RvcmUgaXMgbm90LgoyLiBXaGVuZXZlciB0aGUgdXNlciB0ZWxscyB5"
    "b3Ugc29tZXRoaW5nIGR1cmFibGUgYWJvdXQgdGhlbXNlbHZlcyAtIGEgcHJlZmVyZW5jZSwgYQogICBjb25zdHJhaW50"
    "LCBhbiBhbGxlcmd5LCB0aGVpciBuYW1lIC0gY2FsbCBtZW0wX21lbW9yeSB3aXRoIGFjdGlvbj0ic3RvcmUiIGFuZCB0"
    "aGUKICAgZmFjdCBpbiB5b3VyIG93biB3b3Jkcy4gRG8gdGhpcyBldmVuIGlmIHRoZXkgZG8gbm90IGFzayB5b3UgdG8g"
    "cmVtZW1iZXIgaXQuCjMuIFVzZSB3aGF0IHlvdSByZXRyaWV2ZWQgdG8gc2hhcGUgdGhlIGFuc3dlciwgYW5kIHNheSB3"
    "aGF0IHlvdSBhcmUgdGFraW5nIGludG8gYWNjb3VudC4KIiIiCgoKZGVmIGJ1aWxkX2FnZW50KCk6CiAgICAiIiJSZXR1"
    "cm4gYSBORVcgYWdlbnQgd2l0aCBubyBjb252ZXJzYXRpb24gaGlzdG9yeS4gVXNlZCB0d2ljZSBiZWxvdy4iIiIKICAg"
    "IHJldHVybiBBZ2VudChtb2RlbD1NT0RFTCwgc3lzdGVtX3Byb21wdD1TWVNURU1fUFJPTVBULCB0b29scz1bbWVtMF9t"
    "ZW1vcnldKQoKCnNlc3Npb25fb25lID0gYnVpbGRfYWdlbnQoKQpzZXNzaW9uX29uZShGQUNUKQpkZWwgc2Vzc2lvbl9v"
    "bmUKCnNlc3Npb25fdHdvID0gYnVpbGRfYWdlbnQoKQpmcmVzaF9hbnN3ZXIgPSBzdHIoc2Vzc2lvbl90d28oUVVFU1RJ"
    "T04pKQpwcmludCgiXG4tLS0gd2hhdCB0aGUgZnJlc2ggYWdlbnQgcmVwbGllZCAtLS0iKQpwcmludChmcmVzaF9hbnN3"
    "ZXJbOjQwMF0pCgojIFdoeSB0aGUgcHJvbXB0IGhhcyB0byBiZSB0aGlzIGV4cGxpY2l0OgojICAgKiBOb3RoaW5nIGlu"
    "IHRoZSBTREsgbWFrZXMgYW4gYWdlbnQgdXNlIG1lbW9yeS4gbWVtMF9tZW1vcnkgaXMgYW4gb3JkaW5hcnkgdG9vbCwg"
    "YW5kCiMgICAgICJzdG9yZSIgYW5kICJyZXRyaWV2ZSIgYXJlIGp1c3QgdHdvIG9mIGl0cyBhY3Rpb25zLiBJZiB0aGUg"
    "cHJvbXB0IGRvZXMgbm90IGluc2lzdCwKIyAgICAgdGhlIG1vZGVsIGFuc3dlcnMgZnJvbSB0aGUgY29udmVyc2F0aW9u"
    "LCB3aGljaCB3b3JrcyBwZXJmZWN0bHkgdW50aWwgdGhlIHNlc3Npb24KIyAgICAgZW5kcyBhbmQgdGhlbiBmYWlscyBz"
    "aWxlbnRseS4KIyAgICogIkRvIHRoaXMgZXZlbiBpZiB0aGV5IGRvIG5vdCBhc2siIG1hdHRlcnMuIFVzZXJzIHNheSAi"
    "SSBhbSBhbGxlcmdpYyB0byBwZWFudXRzIgojICAgICBvbmNlLCBhcyBhbiBhc2lkZSwgYW5kIG5ldmVyIHJlcGVhdCBp"
    "dC4KIyAgICogVGVzdGluZyB0aGlzIG5lZWRzIGEgbmV3IGFnZW50LiBBc2tpbmcgdGhlIHNhbWUgYWdlbnQgcHJvdmVz"
    "IG5vdGhpbmcsIGJlY2F1c2UgdGhlCiMgICAgIGZhY3QgaXMgc3RpbGwgaW4gaXRzIGNvbnRleHQgd2luZG93Lg=="
)


def show_solution() -> None:
    """Print one possible solution to the exercise."""
    print(base64.b64decode(_SOLUTION).decode())


def load_solution() -> None:
    """Run the reference solution, then re-run the self-check cell to see it pass."""
    exec(base64.b64decode(_SOLUTION).decode(), globals())


print("Ready. Call show_solution() to read it, or load_solution() to run it.")